# Phase 3 — Đánh giá Stream & Static Fuzzy Baseline Benchmark

Dự án: **An Evolving Fuzzy Reasoning System for Sensor Stream Anomaly Detection under Concept Drift**

Tập tin này độc lập với Phase 1 và Phase 2, kế thừa toàn bộ cấu hình hệ suy diễn mờ Mamdani tĩnh (Static FIS) đã được chốt và đóng băng tại Phase 2:
- 5 biến cảm biến đầu vào: Air temp, Process temp, RPM, Torque, Tool wear.
- 12 luật mờ Mamdani (4 HIGH, 5 MEDIUM, 3 LOW).
- Fuzzification liên tục giải tích (analytical piecewise linear).
- Suy diễn Mamdani: AND = min, Implication = min, Aggregation = max, Defuzzification = centroid.
- Không gian Anomaly Score: $[0, 1]$.
- Phân vùng dữ liệu tuần tự theo Experimental Data Protocol:
  - **Train:** 6.000 mẫu đầu (UDI 1 → 6000)
  - **Validation:** 2.000 mẫu tiếp theo (UDI 6001 → 8000)
  - **Test:** 2.000 mẫu cuối (UDI 8001 → 10000)

In [1]:
# Phase 3.0 — Setup môi trường và nạp phân vùng dữ liệu theo giao thức thực nghiệm
from pathlib import Path
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# Load dữ liệu (hỗ trợ đường dẫn tương đối linh hoạt)
data_path = Path("../data/ai4i2020.csv") if Path("../data/ai4i2020.csv").exists() else Path("data/ai4i2020.csv")
df = pd.read_csv(data_path)

sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# Phân vùng dữ liệu tuần tự (Sequential Split) theo Protocol Phase 1.5
train_df = df.iloc[:6000].copy()
val_df   = df.iloc[6000:8000].copy()
test_df  = df.iloc[8000:].copy()

print(f"Tổng dataset : {len(df)} mẫu")
print(f"Train set    : {len(train_df)} mẫu (UDI {train_df['UDI'].min()} → {train_df['UDI'].max()})")
print(f"Validation   : {len(val_df)} mẫu (UDI {val_df['UDI'].min()} → {val_df['UDI'].max()})")
print(f"Test set     : {len(test_df)} mẫu (UDI {test_df['UDI'].min()} → {test_df['UDI'].max()})")

Tổng dataset : 10000 mẫu
Train set    : 6000 mẫu (UDI 1 → 6000)
Validation   : 2000 mẫu (UDI 6001 → 8000)
Test set     : 2000 mẫu (UDI 8001 → 10000)


In [2]:
# Phase 3.0 — Đóng băng cấu hình Static Mamdani FIS từ Phase 2

# 1. Tham số Membership Functions của 5 biến cảm biến đầu vào
mf_params = {
    "air_temp": {
        "LOW":    ("trap", [295.3, 295.3, 298.0, 300.4]),
        "MEDIUM": ("tri",  [298.0, 300.4, 302.5]),
        "HIGH":   ("trap", [300.4, 302.5, 304.5, 304.5])
    },
    "process_temp": {
        "LOW":    ("trap", [305.7, 305.7, 308.0, 309.7]),
        "MEDIUM": ("tri",  [308.0, 309.7, 311.5]),
        "HIGH":   ("trap", [309.7, 311.5, 313.8, 313.8])
    },
    "rpm": {
        "LOW":    ("trap", [1168.0, 1168.0, 1350.0, 1500.0]),
        "MEDIUM": ("tri",  [1350.0, 1504.0, 1800.0]),
        "HIGH":   ("trap", [1600.0, 1880.0, 2886.0, 2886.0])
    },
    "torque": {
        "LOW":    ("trap", [3.8, 3.8, 25.0, 40.0]),
        "MEDIUM": ("tri",  [25.0, 40.0, 55.0]),
        "HIGH":   ("trap", [40.0, 55.0, 76.2, 76.2])
    },
    "tool_wear": {
        "LOW":    ("trap", [0.0, 0.0, 54.0, 109.0]),
        "MEDIUM": ("tri",  [54.0, 109.0, 164.0]),
        "HIGH":   ("trap", [109.0, 164.0, 253.0, 253.0])
    }
}

# 2. Universe và MF của biến đầu ra Anomaly Score
anomaly_universe = np.linspace(0, 1, 1000)
anomaly_mfs = {
    "LOW":    fuzz.trapmf(anomaly_universe, [0.0, 0.0, 0.25, 0.50]),
    "MEDIUM": fuzz.trimf( anomaly_universe, [0.25, 0.50, 0.75]),
    "HIGH":   fuzz.trapmf(anomaly_universe, [0.50, 0.75, 1.0, 1.0])
}

# 3. Hàm tính toán mức thuộc liên tục giải tích
def eval_mf(x, mf_type, params):
    x = float(x)
    if mf_type == "tri":
        a, b, c = params
        if a < b and a <= x <= b:
            return (x - a) / (b - a)
        elif b < c and b <= x <= c:
            return (c - x) / (c - b)
        elif x == b:
            return 1.0
        return 0.0
    elif mf_type == "trap":
        a, b, c, d = params
        if x < a:
            return 1.0 if a == b else 0.0
        elif a <= x < b:
            return (x - a) / (b - a) if b > a else 1.0
        elif b <= x <= c:
            return 1.0
        elif c < x <= d:
            return (d - x) / (d - c) if d > c else 1.0
        else:
            return 1.0 if c == d else 0.0

def compute_memberships(sample):
    mapping = {
        "air_temp":     sample["Air temperature [K]"],
        "process_temp": sample["Process temperature [K]"],
        "rpm":          sample["Rotational speed [rpm]"],
        "torque":       sample["Torque [Nm]"],
        "tool_wear":    sample["Tool wear [min]"],
    }
    m = {}
    for var_name, val in mapping.items():
        m[var_name] = {}
        for term, (m_type, params) in mf_params[var_name].items():
            m[var_name][term] = eval_mf(val, m_type, params)
    return m

# 4. Hệ 12 luật mờ Mamdani
rules = [
    # 🔴 HIGH anomaly — 4 rules
    ("R1", [("rpm", "LOW"), ("torque", "HIGH"), ("air_temp", "HIGH")], "HIGH"),
    ("R2", [("torque", "HIGH"), ("air_temp", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),
    ("R3", [("rpm", "LOW"), ("torque", "HIGH"), ("process_temp", "HIGH")], "HIGH"),
    ("R4", [("rpm", "LOW"), ("torque", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),

    # 🟡 MEDIUM anomaly — 5 rules
    ("R5", [("rpm", "LOW"), ("torque", "HIGH")], "MEDIUM"),
    ("R6", [("rpm", "LOW"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R7", [("torque", "HIGH"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R8", [("torque", "HIGH"), ("tool_wear", "HIGH")], "MEDIUM"),
    ("R9", [("torque", "HIGH"), ("process_temp", "HIGH")], "MEDIUM"),

    # 🟢 LOW anomaly — 3 rules
    ("R10", [("rpm", "MEDIUM"), ("torque", "MEDIUM")], "LOW"),
    ("R11", [("rpm", "MEDIUM"), ("torque", "LOW")], "LOW"),
    ("R12", [("rpm", "HIGH"), ("torque", "LOW")], "LOW"),
]

# 5. Động cơ suy diễn Mamdani
def mamdani_inference(sample, return_details=False):
    mu = compute_memberships(sample)
    rule_activations = {}
    implied_outputs = []
    
    for r_id, antecedents, consequent in rules:
        alpha = min(mu[var][term] for var, term in antecedents)
        rule_activations[r_id] = alpha
        implied_mf = np.fmin(alpha, anomaly_mfs[consequent])
        implied_outputs.append(implied_mf)
        
    aggregated_mf = np.zeros_like(anomaly_universe)
    for mf in implied_outputs:
        aggregated_mf = np.fmax(aggregated_mf, mf)
        
    if np.sum(aggregated_mf) == 0:
        score = 0.0
    else:
        score = fuzz.defuzz(anomaly_universe, aggregated_mf, "centroid")
        
    if return_details:
        return score, rule_activations, mu, aggregated_mf
    return score, rule_activations, mu

print("Static Mamdani FIS Engine và 12 luật đã sẵn sàng.")

Static Mamdani FIS Engine và 12 luật đã sẵn sàng.


---
### Phase 3.1 — Chạy FIS trên toàn bộ tập dữ liệu Train (6.000 mẫu)

Khảo sát phân phối Anomaly Score của tập Train để kiểm tra tính phân tách giữa hai lớp Normal và Failure trước khi chuyển sang tập Validation.

In [3]:
# PHASE 3.1 — Generate anomaly scores for entire Train set

train_scores = []

for idx, sample in train_df.iterrows():
    score, _, _ = mamdani_inference(sample)
    train_scores.append(score)

train_eval_df = train_df.copy()
train_eval_df["Anomaly Score"] = train_scores

print("Number of samples:", len(train_eval_df))
print("Score range:",
      round(train_eval_df["Anomaly Score"].min(), 4),
      "→",
      round(train_eval_df["Anomaly Score"].max(), 4))

print("\nMean score by actual class:")
print(
    train_eval_df
    .groupby("Machine failure")["Anomaly Score"]
    .agg(["count", "mean", "median", "min", "max"])
)

Number of samples: 6000
Score range: 0.0 → 0.6898

Mean score by actual class:
                 count      mean    median       min       max
Machine failure                                               
0                 5745  0.320477  0.225615  0.000000  0.689815
1                  255  0.558070  0.644626  0.194445  0.689815


---
### Phase 3.2 — Chạy FIS trên tập Validation (2.000 mẫu)

Áp dụng hệ suy diễn mờ Mamdani đã đóng băng trên tập Validation (UDI 6001 → 8000) để khảo sát phân phối điểm số của hai lớp Normal và Failure trước khi lựa chọn ngưỡng quyết định $\tau$.

In [4]:
# PHASE 3.2 — Generate anomaly scores for Validation set

val_scores = []

for idx, sample in val_df.iterrows():
    score, _, _ = mamdani_inference(sample)
    val_scores.append(score)

val_eval_df = val_df.copy()
val_eval_df["Anomaly Score"] = val_scores

print("Number of samples:", len(val_eval_df))

print(
    "Score range:",
    round(val_eval_df["Anomaly Score"].min(), 4),
    "→",
    round(val_eval_df["Anomaly Score"].max(), 4)
)

print("\nMean score by actual class:")
print(
    val_eval_df
    .groupby("Machine failure")["Anomaly Score"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nQuantiles by actual class:")
print(
    val_eval_df
    .groupby("Machine failure")["Anomaly Score"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95])
    .unstack()
)

Number of samples: 2000
Score range: 0.1944 → 0.6898

Mean score by actual class:
                 count      mean    median       min       max
Machine failure                                               
0                 1955  0.338487  0.233116  0.194445  0.689815
1                   45  0.525178  0.640925  0.194445  0.689815

Quantiles by actual class:
                     0.25      0.50      0.75      0.90      0.95
Machine failure                                                  
0                0.212559  0.233116  0.463213  0.620025  0.659937
1                0.227959  0.640925  0.679176  0.689815  0.689815


---
### Phase 3.3 — Quét ngưỡng quyết định $\tau$ trên tập Validation (Threshold Sweep)

Khảo sát sự biến thiên của Precision, Recall, F1-score và ma trận nhầm lẫn (TP, FP, FN, TN) khi thay đổi ngưỡng $\tau \in [0.20, 0.69]$ trên tập Validation nhằm tìm kiếm các ngưỡng cân bằng tối ưu.

In [5]:
# PHASE 3.3 — Threshold sweep on Validation

from sklearn.metrics import precision_score, recall_score, f1_score

y_val = val_eval_df["Machine failure"].to_numpy()
scores_val = val_eval_df["Anomaly Score"].to_numpy()

thresholds = np.arange(0.20, 0.691, 0.01)

threshold_results = []

for tau in thresholds:
    y_pred = (scores_val >= tau).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    tp = np.sum((y_val == 1) & (y_pred == 1))
    fp = np.sum((y_val == 0) & (y_pred == 1))
    fn = np.sum((y_val == 1) & (y_pred == 0))
    tn = np.sum((y_val == 0) & (y_pred == 0))

    threshold_results.append({
        "Threshold": tau,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(threshold_results)

print("Top thresholds by F1:")
print(
    threshold_df
    .sort_values("F1", ascending=False)
    .head(10)
    .to_string(index=False)
)

Top thresholds by F1:
 Threshold  Precision   Recall       F1  TP  FP  FN   TN
      0.67   0.168539 0.333333 0.223881  15  74  30 1881
      0.64   0.138728 0.533333 0.220183  24 149  21 1806
      0.68   0.177419 0.244444 0.205607  11  51  34 1904
      0.65   0.133333 0.444444 0.205128  20 130  25 1825
      0.66   0.140351 0.355556 0.201258  16  98  29 1857
      0.63   0.123711 0.533333 0.200837  24 170  21 1785
      0.62   0.113122 0.555556 0.187970  25 196  20 1759
      0.61   0.104000 0.577778 0.176271  26 224  19 1731
      0.60   0.101124 0.600000 0.173077  27 240  18 1715
      0.59   0.092150 0.600000 0.159763  27 266  18 1689


---
### Phase 3.4 — Khảo sát chi tiết hành vi ngưỡng trong vùng quyết định chính $[0.50, 0.70]$

Kiểm tra toàn bộ các giá trị ngưỡng từ $0.50$ đến $0.69$ để quan sát mức độ ổn định của $F_1$-score và sự biến thiên của Recall và Precision trước khi đưa ra quyết định đóng băng $\tau$.

In [6]:
# PHASE 3.4 — Inspect threshold behavior in the main decision region

print(
    threshold_df[
        (threshold_df["Threshold"] >= 0.50) &
        (threshold_df["Threshold"] <= 0.70)
    ].to_string(index=False)
)

 Threshold  Precision   Recall       F1  TP  FP  FN   TN
      0.50   0.068670 0.711111 0.125245  32 434  13 1521
      0.51   0.072072 0.711111 0.130879  32 412  13 1543
      0.52   0.075117 0.711111 0.135881  32 394  13 1561
      0.53   0.074879 0.688889 0.135076  31 383  14 1572
      0.54   0.078680 0.688889 0.141230  31 363  14 1592
      0.55   0.079365 0.666667 0.141844  30 348  15 1607
      0.56   0.085227 0.666667 0.151134  30 322  15 1633
      0.57   0.087349 0.644444 0.153846  29 303  16 1652
      0.58   0.088608 0.622222 0.155125  28 288  17 1667
      0.59   0.092150 0.600000 0.159763  27 266  18 1689
      0.60   0.101124 0.600000 0.173077  27 240  18 1715
      0.61   0.104000 0.577778 0.176271  26 224  19 1731
      0.62   0.113122 0.555556 0.187970  25 196  20 1759
      0.63   0.123711 0.533333 0.200837  24 170  21 1785
      0.64   0.138728 0.533333 0.220183  24 149  21 1806
      0.65   0.133333 0.444444 0.205128  20 130  25 1825
      0.66   0.140351 0.355556 

---
### Phase 3.5 — Đóng băng ngưỡng quyết định (Freeze Static Fuzzy Threshold)

> **Nguyên tắc phương pháp luận:**
> *"Using the predefined validation criterion of maximum F1-score, the decision threshold was selected as $\tau = 0.67$ on the validation set and subsequently fixed for test evaluation."*
> 
> Giá trị $\tau = 0.67$ được đóng băng tuyệt đối, không điều chỉnh thêm trước khi chuyển sang đánh giá trên tập Test.

In [7]:
# PHASE 3.5 — Freeze Static Fuzzy threshold

TAU_STATIC = 0.67

best_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

print("Frozen threshold for Static Fuzzy:")
print(f"tau = {TAU_STATIC:.2f}")

print("\nValidation performance at frozen threshold:")

for col in ["Precision", "Recall", "F1", "TP", "FP", "FN", "TN"]:
    print(f"{col}: {best_row[col]}")

Frozen threshold for Static Fuzzy:
tau = 0.67

Validation performance at frozen threshold:
Precision: 0.16853932584269662
Recall: 0.3333333333333333
F1: 0.22388059701492538
TP: 15.0
FP: 74.0
FN: 30.0
TN: 1881.0


---
### Phase 3.6 — Đánh giá Static Fuzzy Baseline trên tập Test (2.000 mẫu cuối)

Đánh giá hiệu năng của hệ cơ sở mờ tĩnh trên tập Test độc lập (UDI 8001 → 10000) với ngưỡng $\tau = 0.67$ đã được đóng băng từ tập Validation.

In [8]:
# PHASE 3.6 — Static Fuzzy evaluation on Test

test_scores = []

for idx, sample in test_df.iterrows():
    score, _, _ = mamdani_inference(sample)
    test_scores.append(score)

test_eval_df = test_df.copy()
test_eval_df["Anomaly Score"] = test_scores

# Fixed threshold selected on Validation
test_eval_df["Prediction"] = (
    test_eval_df["Anomaly Score"] >= TAU_STATIC
).astype(int)

# Ground truth
y_test = test_eval_df["Machine failure"].to_numpy()
y_pred_test = test_eval_df["Prediction"].to_numpy()

# Metrics
precision_test = precision_score(
    y_test, y_pred_test, zero_division=0
)

recall_test = recall_score(
    y_test, y_pred_test, zero_division=0
)

f1_test = f1_score(
    y_test, y_pred_test, zero_division=0
)

# Confusion matrix components
tp_test = np.sum((y_test == 1) & (y_pred_test == 1))
fp_test = np.sum((y_test == 0) & (y_pred_test == 1))
fn_test = np.sum((y_test == 1) & (y_pred_test == 0))
tn_test = np.sum((y_test == 0) & (y_pred_test == 0))

print("Static Fuzzy — Test Evaluation")
print("--------------------------------")

print(f"Test samples: {len(test_eval_df)}")
print(f"Threshold: {TAU_STATIC:.2f}")

print("\nScore range:")
print(
    round(test_eval_df["Anomaly Score"].min(), 4),
    "→",
    round(test_eval_df["Anomaly Score"].max(), 4)
)

print("\nConfusion Matrix:")
print(f"TP: {tp_test}")
print(f"FP: {fp_test}")
print(f"FN: {fn_test}")
print(f"TN: {tn_test}")

print("\nMetrics:")
print(f"Precision: {precision_test:.4f}")
print(f"Recall:    {recall_test:.4f}")
print(f"F1:        {f1_test:.4f}")

Static Fuzzy — Test Evaluation
--------------------------------
Test samples: 2000
Threshold: 0.67

Score range:
0.0 → 0.6898

Confusion Matrix:
TP: 13
FP: 34
FN: 26
TN: 1927

Metrics:
Precision: 0.2766
Recall:    0.3333
F1:        0.3023
